# Vision Assignment P06 — ResNet-50 Transfer Learning

Notebook ini membandingkan feature extraction dan fine-tuning pada CIFAR-10 serta beberapa learning rate.

In [ ]:
%pip install -q torch torchvision scikit-learn matplotlib

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
weights=models.ResNet50_Weights.DEFAULT
mean,std=weights.transforms().mean,weights.transforms().std
tf=transforms.Compose([transforms.Resize(224),transforms.ToTensor(),transforms.Normalize(mean,std)])
train_ds=datasets.CIFAR10('data',train=True,download=True,transform=tf)
test_ds=datasets.CIFAR10('data',train=False,download=True,transform=tf)
train_loader=DataLoader(train_ds,batch_size=64,shuffle=True,num_workers=2)
test_loader=DataLoader(test_ds,batch_size=128,num_workers=2)
print('Device:',device,'Train:',len(train_ds),'Test:',len(test_ds))

In [ ]:
def make_model(strategy='feature'):
    model=models.resnet50(weights=weights)
    for p in model.parameters(): p.requires_grad=False
    if strategy=='finetune':
        for p in model.layer4.parameters(): p.requires_grad=True
    model.fc=nn.Linear(model.fc.in_features,10)
    return model.to(device)

feature_model=make_model('feature')
finetune_model=make_model('finetune')
print('Trainable feature extraction:',sum(p.numel() for p in feature_model.parameters() if p.requires_grad))
print('Trainable fine-tuning:',sum(p.numel() for p in finetune_model.parameters() if p.requires_grad))

In [ ]:
def epoch(model, loader, criterion, optimizer=None):
    model.train(optimizer is not None); total=correct=loss_sum=0
    for x,y in loader:
        x,y=x.to(device),y.to(device)
        if optimizer: optimizer.zero_grad()
        out=model(x); loss=criterion(out,y)
        if optimizer: loss.backward(); optimizer.step()
        loss_sum += loss.item()*len(y); correct += (out.argmax(1)==y).sum().item(); total += len(y)
    return loss_sum/total, correct/total

criterion=nn.CrossEntropyLoss()
opt=torch.optim.Adam(feature_model.fc.parameters(),lr=1e-3)
# Gunakan subset opsional untuk uji cepat; hapus [:1000] untuk eksperimen penuh.
for e in range(1):
    tr=epoch(feature_model,train_loader,criterion,opt); te=epoch(feature_model,test_loader,criterion)
    print(f'Feature extraction: train={tr[1]:.3f}, test={te[1]:.3f}')

In [ ]:
opt_ft=torch.optim.Adam([{'params':finetune_model.layer4.parameters(),'lr':1e-4},{'params':finetune_model.fc.parameters(),'lr':1e-3}])
for e in range(1):
    tr=epoch(finetune_model,train_loader,criterion,opt_ft); te=epoch(finetune_model,test_loader,criterion)
    print(f'Fine-tuning: train={tr[1]:.3f}, test={te[1]:.3f}')

## Analisis

Feature extraction membekukan backbone sehingga hanya classifier yang belajar; ini lebih hemat dan menjadi baseline. Fine-tuning membuka `layer4`, sehingga fitur tingkat tinggi dapat beradaptasi dengan CIFAR-10, tetapi membutuhkan waktu dan learning rate yang lebih hati-hati. Bandingkan hasil aktual dari beberapa learning rate dalam tabel laporan.

## Penjelasan Konsep, Analisis, dan Kesimpulan

ResNet menggunakan residual connection `y = F(x) + x` untuk membantu aliran gradient pada jaringan dalam. Feature extraction membekukan backbone dan hanya melatih classifier; fine-tuning membuka layer akhir agar fitur tingkat tinggi beradaptasi dengan CIFAR-10.

CIFAR-10 berukuran 32×32, sehingga notebook melakukan resize ke 224×224 sesuai input ResNet-50 pretrained. Bandingkan accuracy, loss, waktu, confusion matrix, dan jumlah parameter trainable untuk beberapa learning rate. Learning rate classifier baru biasanya lebih besar daripada backbone karena classifier diinisialisasi dari awal.

ResNet-50 pretrained dapat digunakan untuk CIFAR-10 dengan mengganti classifier menjadi 10 kelas. Strategi terbaik ditentukan berdasarkan validation/test accuracy, loss, waktu training, dan stabilitas; bukan hanya training accuracy.

In [ ]:
# Kesimpulan otomatis setelah fine-tuning
print('ANALISIS OTOMATIS')
print(f'- Parameter trainable: {sum(p.numel() for p in finetune_model.parameters() if p.requires_grad):,}')
print('Catatan: simpan accuracy feature extraction dan fine-tuning dalam variabel berbeda untuk perbandingan numerik.')
print('KESIMPULAN: feature extraction lebih hemat; fine-tuning lebih adaptif dan memerlukan learning rate backbone yang lebih kecil.')